# ARGUS Agent 5a: Qwen3-8B Colab validation

Run this notebook in a fresh Google Colab GPU runtime. It validates Qwen3-8B generation and exports an INT4 OpenVINO model. **Do not paste credentials into notebook cells.** If Hugging Face authentication is required, use the browser prompt from `huggingface_hub.login()`.

## Runtime setup

Choose **Runtime > Change runtime type > T4 GPU**, save, and run the next cell. After the install cell, use **Runtime > Restart session** before continuing.

In [ ]:
!nvidia-smi

In [ ]:
# Remove Colab's conflicting preinstalled distributions.
!pip uninstall -y optimum optimum-intel diffusers huggingface_hub transformers
!pip install --no-cache-dir -U "huggingface_hub>=0.34.0" "transformers>=4.51.0" accelerate "optimum-intel[openvino]" openvino sentencepiece
print("INSTALL COMPLETE: now select Runtime > Restart session, then continue.")

In [ ]:
import importlib.metadata as metadata
import torch
import transformers
import huggingface_hub
import openvino

for package in ("transformers", "huggingface-hub", "accelerate", "optimum", "optimum-intel", "openvino"):
    try:
        print(package, metadata.version(package))
    except metadata.PackageNotFoundError:
        print(package, "NOT INSTALLED")
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("OpenVINO:", openvino.__version__)
!pip check

In [ ]:
from huggingface_hub import login

# Run only if the model download asks for authentication. A secure browser prompt opens.
# login()

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
print("Loaded:", MODEL_ID)
print("Device:", next(model.parameters()).device)

In [ ]:
prompt = """Agent 5a threat-intelligence test.
Validated evidence: finding_id=F-1, fact=Connection to 192.0.2.10.
External feed: historical report mentioning 192.0.2.10; current_case_confirmation=false.
Explain the enrichment and state whether the feed proves current-case activity.
Do not invent facts."""
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=160, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

In [ ]:
!rm -rf /content/Qwen3-8B-openvino-int4
!optimum-cli export openvino --model Qwen/Qwen3-8B --task text-generation-with-past --trust-remote-code --weight-format int4 --group-size 128 --sym /content/Qwen3-8B-openvino-int4

In [ ]:
from pathlib import Path
import openvino as ov

export_dir = Path("/content/Qwen3-8B-openvino-int4")
xml_files = list(export_dir.rglob("*.xml"))
bin_files = list(export_dir.rglob("*.bin"))
print("XML:", xml_files)
print("BIN:", bin_files)
assert xml_files and bin_files, "OpenVINO IR files were not produced"
model_xml = export_dir / "openvino_model.xml"
assert model_xml.exists(), f"Missing model IR: {model_xml}"

core = ov.Core()
print("OpenVINO devices:", core.available_devices)
compiled = core.compile_model(str(model_xml), "CPU")
print("Compiled on CPU:", compiled is not None)
if "GPU" in core.available_devices:
    core.compile_model(str(model_xml), "GPU")
    print("Compiled on GPU: True")

import shutil
archive = shutil.make_archive("/content/Qwen3-8B-openvino-int4", "zip", export_dir)
print("Download artifact:", archive)

## Expected completion criteria

- CUDA is `True` and a GPU name is printed.
- Qwen3-8B loads and produces a response.
- The export creates at least one `.xml` and one `.bin` file.
- OpenVINO compiles the exported IR on CPU; GPU compilation is attempted only when available.
- Download `/content/Qwen3-8B-openvino-int4.zip` from the Colab file browser and place it under `D:\ARGUS\models` locally.